In [0]:
dbutils.widgets.text("catalog","claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
df = spark.table(f"{catalog}.bronze.policy_change_cdc")

In [0]:
df.display()

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.silver.dim_policy_scd2(
        member_id string,
        effective_from date,
        op string,
        op_ts timestamp,
        plan_code string,
        policy_id string,
        end_date DATE,
        is_current BOOLEAN
    )
    using delta
""")

In [0]:
from pyspark.sql.functions import *;
from pyspark.sql.window import Window

df = df.withColumn("effective_from",to_date("effective_from","yyyy-MM-dd")) \
    .withColumn("member_id",col("member_id")) \
    .withColumn("plan_code",lower(trim("plan_code"))) \
    .withColumn("op",upper(trim("op"))) \
    .withColumn("op_ts",to_timestamp("op_ts","yyyy-MM-dd'T'HH:mm:ss"))

df = df.drop("ingested_at","src_file")

# Deduplicate source first so we only look at the absolute latest record per member
window_spec = Window.partitionBy("member_id").orderBy(col("op_ts").desc(), col("effective_from").desc())

df = df.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn") 
    
df.createOrReplaceTempView("policy_change_view")

In [0]:
# Step 1: Expire old records
spark.sql(f"""
MERGE INTO {catalog}.silver.dim_policy_scd2 AS t
USING policy_change_view AS s
ON t.member_id = s.member_id AND t.is_current = true
WHEN MATCHED AND t.plan_code != s.plan_code THEN
  UPDATE SET t.is_current = false, t.end_date = s.effective_from
""")

# Step 2: Insert new records
spark.sql(f"""
MERGE INTO {catalog}.silver.dim_policy_scd2 AS t
USING policy_change_view AS s
ON t.member_id = s.member_id AND t.is_current = true
WHEN NOT MATCHED THEN
  INSERT (member_id, effective_from, op, op_ts, plan_code, policy_id, end_date, is_current)
  VALUES (s.member_id, s.effective_from, s.op, s.op_ts, s.plan_code, s.policy_id, NULL, true)
""")


In [0]:
# # Step 1: Expire old active records where a plan change is detected
# spark.sql(f"""
# MERGE INTO {catalog}.silver.policy_change AS t
# USING (
#     WITH ranked_source AS (
#         SELECT *, 
#                ROW_NUMBER() OVER(PARTITION BY member_id ORDER BY op_ts DESC, effective_from DESC) as rn
#         FROM policy_change_view
#     )
#     SELECT * FROM ranked_source WHERE rn = 1
# ) AS s
# ON t.member_id = s.member_id 
# AND t.is_current = true
# WHEN MATCHED AND t.plan_code != s.plan_code THEN
#   UPDATE SET 
#     t.is_current = false,
#     t.end_date = s.effective_from
# """)

# # Step 2: Insert the new active records (both brand new members and updated plans)
# spark.sql(f"""
# MERGE INTO {catalog}.silver.policy_change AS t
# USING (
#     WITH ranked_source AS (
#         SELECT *, 
#                ROW_NUMBER() OVER(PARTITION BY member_id ORDER BY op_ts DESC, effective_from DESC) as rn
#         FROM policy_change_view
#     )
#     SELECT * FROM ranked_source WHERE rn = 1
# ) AS s
# ON t.member_id = s.member_id 
# AND t.is_current = true
# WHEN NOT MATCHED THEN
#   INSERT (
#     member_id,
#     effective_from,
#     op,
#     op_ts,
#     plan_code,
#     policy_id,
#     end_date,
#     is_current
#   )
#   VALUES (
#     s.member_id,
#     s.effective_from,
#     s.op,
#     s.op_ts,
#     s.plan_code,
#     s.policy_id,
#     NULL,
#     true
#   )
# """)


In [0]:
row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.silver.dim_policy_scd2")
display(row_count)
#1run -3

In [0]:
cc = spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.silver.claims_clean")
display(cc)

In [0]:
cq = spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.silver.claims_quarantine")
display(cq)